In [1]:
# ===== 1: 설정 =====
# 첫 파인튜닝. 학습 형식 = RAG 추론 형식(data/prompts.py). 샘플은 data/finetune_data.py(C안)가 만든다:
#   정답 발췌 + 방해 발췌 70% / 정답 발췌만 20% / 방해 발췌만 → "확인되지 않음" 10%, val은 unit_id 단위 분할.
# 이전 시도에서 확인한 것(설계 근거, 옛 청크·옛 FAQ로 한 별개 작업이라 수치는 참조하지 않음):
#   - 발췌 없이 질문→답만 학습하면 사실을 외우려 한다(발췌 없는 질문에 원문에 없는 기간을 단정)
#   - "발췌만 근거로 답하라"는 지시와 발췌 없는 데이터가 모순이면 지시가 무시된다
#   - 고정 패딩(max_length)으로 계산의 약 2/3가 패딩에 쓰였다 → 동적 패딩
import json
import os
import sys

from collections import Counter
from pathlib import Path

PROJECT_DIR = Path.home() / "AI_programming" / "petmint-rag-chatbot"
sys.path.insert(0, str(PROJECT_DIR))
from data import prompts

BASE_MODEL_DIR = PROJECT_DIR / "models" / "base" / "Tri-7B"
# 전체 재구축(data/rebuild.py)은 환경변수로 학습 데이터 폴더와 어댑터 이름 꼬리표를 넘긴다(학습 설정은 그대로). 없으면 기존 값
DATA_DIR = Path(os.environ.get("PETMINT_FT_DATA_DIR", PROJECT_DIR / "data" / "finetune"))
ADAPTER_DIR = PROJECT_DIR / "models" / "lora_adapter"
ADAPTER_TAG = os.environ.get("PETMINT_ADAPTER_TAG", "")

MAX_LENGTH = 3072   # data/finetune_data.MAX_LENGTH와 같아야 한다(샘플이 이미 이 안에 맞춰져 있음)
MAX_STEPS = None    # 20이면 첫 20 step만 돈다(메모리·시간 확인용). None이면 전체 학습

train_records = [json.loads(l) for l in open(DATA_DIR / "train.jsonl", encoding="utf-8")]
val_records = [json.loads(l) for l in open(DATA_DIR / "val.jsonl", encoding="utf-8")]
stats = json.load(open(DATA_DIR / "stats.json", encoding="utf-8"))
assert stats["prompt_version"] == prompts.PROMPT_VERSION, "샘플과 prompts.py 버전이 다르다 — finetune_data를 다시 돌릴 것"
assert stats["max_length"] == MAX_LENGTH
assert stats["unit_overlap"] == 0, "train·val에 같은 단위가 있다"

output_dir = ADAPTER_DIR / f"{prompts.PROMPT_VERSION}_{len(train_records)}건{'_' + ADAPTER_TAG if ADAPTER_TAG else ''}"
print("베이스 모델:", BASE_MODEL_DIR, BASE_MODEL_DIR.exists())
print(f"train {len(train_records)}건({stats['train_units']}단위), val {len(val_records)}건({stats['val_units']}단위)")
print("샘플 종류(train):", Counter(r["meta"]["kind"] for r in train_records))
print("출력 경로:", output_dir)

베이스 모델: /home/kopo17/AI_programming/petmint-rag-chatbot/models/base/Tri-7B True
train 4811건(511단위), val 532건(55단위)
샘플 종류(train): Counter({'distract': 3164, 'gold_only': 1163, 'no_answer': 484})
출력 경로: /home/kopo17/AI_programming/petmint-rag-chatbot/models/lora_adapter/rag-1_4811건


In [2]:
# ===== 2: 토큰화 — loss는 답변 토큰에만, 패딩은 배치 단위(동적) =====
import statistics
from datasets import Dataset
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_DIR)


def encode(messages: list[dict]) -> dict:
    full = tokenizer.apply_chat_template(messages, tokenize=False)
    prompt = tokenizer.apply_chat_template(messages[:-1], tokenize=False, add_generation_prompt=True)
    assert full.startswith(prompt)
    ids = tokenizer(full, add_special_tokens=False)["input_ids"]
    prompt_ids = tokenizer(prompt, add_special_tokens=False)["input_ids"]
    assert ids[:len(prompt_ids)] == prompt_ids, "프롬프트 경계에서 토큰이 합쳐졌다"
    # 발췌·질문(프롬프트)은 -100 — 여기에 loss를 걸면 약관 원문을 외우는 학습이 된다
    labels = [-100] * len(prompt_ids) + ids[len(prompt_ids):]
    return {"input_ids": ids, "attention_mask": [1] * len(ids), "labels": labels, "length": len(ids)}


def build_dataset(records: list[dict]) -> Dataset:
    rows = [encode(r["messages"]) for r in records]
    over = sum(1 for r in rows if r["length"] > MAX_LENGTH)
    assert over == 0, f"MAX_LENGTH 초과 {over}건 — finetune_data 예산과 어긋남"
    return Dataset.from_list(rows)


train_dataset = build_dataset(train_records)
val_dataset = build_dataset(val_records)

for name, ds in [("train", train_dataset), ("val", val_dataset)]:
    L = sorted(ds["length"])
    ans = [sum(1 for x in lab if x != -100) for lab in ds["labels"]]
    print(f"{name}: {len(L)}건 | 길이 평균 {statistics.mean(L):.0f} / 중앙 {L[len(L)//2]} / p90 {L[int(.9*len(L))]} / 최대 {L[-1]}"
          f" | 답변 토큰 평균 {statistics.mean(ans):.0f} | 총 {sum(L):,}토큰(loss {sum(ans):,})")

/home/kopo17/llm_env/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


train: 4811건 | 길이 평균 1209 / 중앙 1073 / p90 2057 / 최대 3064 | 답변 토큰 평균 45 | 총 5,814,954토큰(loss 214,305)
val: 532건 | 길이 평균 1334 / 중앙 1178 / p90 2264 / 최대 3060 | 답변 토큰 평균 48 | 총 709,608토큰(loss 25,567)


In [3]:
# ===== 3: 모델 로드 + QLoRA 설정 =====
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_DIR,
    quantization_config=bnb_config,
    device_map="auto",
    dtype=torch.bfloat16,
)
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)
model.enable_input_require_grads()
model.print_trainable_parameters()

Loading weights: 100%|██████████| 291/291 [00:02<00:00, 101.43it/s]


trainable params: 16,777,216 || all params: 7,543,721,984 || trainable%: 0.2224


In [4]:
# ===== 4: 학습 =====
# batch 1 × accum 32: 최장 샘플(MAX_LENGTH)도 한 번에 1개만 올라가게. 첫 20 step 실측 평균 43s/step(긴 묶음이 앞에 몰림, 앞 5 step 64~66s)
#   → 전체 약 300 step(2 epoch) 3.5~4시간 + eval 6회. 동적 패딩(DataCollatorForSeq2Seq)이라 패딩 낭비가 없다.
# group_by_length: 비슷한 길이끼리 묶는다(batch>1일 때 패딩 절약). 가장 긴 묶음을 맨 앞에 두므로 OOM이 첫 step에서 드러난다.
import time
import torch.nn.functional as F
from transformers import DataCollatorForSeq2Seq, Trainer, TrainerCallback, TrainingArguments

data_collator = DataCollatorForSeq2Seq(tokenizer, padding=True, label_pad_token_id=-100, pad_to_multiple_of=8)

# logits는 답변 구간만 계산한다(2026-10-13: 최장 묶음 첫 step GPU 최대 할당 14.1 → 9.9GiB, loss 동일 — 상대오차 ≤1.5e-6).
# loss는 답변 토큰(평균 45개)에만 걸리는데, 그대로 두면 시퀀스 전체(최대 3,064위치) ×
# 어휘 128,256개의 fp32 logits(한 벌 약 1.6GB, softmax·gradient까지 2~3벌)를 만든다 — 첫 실행 메모리 부족의 주범.
# 답변은 항상 시퀀스 끝(assistant가 마지막 메시지)이라 forward의 logits_to_keep으로 끝 K위치만 lm_head를 통과시킨다.
def collate(features):
    batch = data_collator(features)
    first = (batch["labels"] != -100).float().argmax(dim=1)          # 샘플별 첫 답변 토큰 위치
    batch["logits_to_keep"] = int(batch["labels"].shape[1] - first.min()) + 1   # +1: 첫 답변 토큰을 예측하는 직전 위치
    return batch


def tail_loss(outputs, labels, num_items_in_batch=None):
    """HF 기본 causal LM loss와 같은 정의(다음 토큰 예측, -100 무시, 누적 배치 전체 답변 토큰 수로 나눔)를 끝 K위치에서만.
    compute_loss_func가 있으면 Trainer가 labels를 빼서 넘기고 accum 나눗셈을 하지 않는다(num_items_in_batch로 나눌 것)."""
    logits = outputs.logits                       # [B, K, V]
    target = labels[:, -logits.shape[1]:][:, 1:]  # 위치 t의 logits → 라벨 t+1
    loss = F.cross_entropy(logits[:, :-1].float().reshape(-1, logits.shape[-1]), target.reshape(-1).to(logits.device),
                           ignore_index=-100, reduction="sum")
    n = num_items_in_batch if num_items_in_batch is not None else (target != -100).sum()
    return loss / n


class PrintLossCallback(TrainerCallback):
    """loss와 함께 step 시간·GPU 최대 메모리를 출력(첫 20 step 확인용)"""
    def on_train_begin(self, args, state, control, **kwargs):
        self.t0 = time.time()
        torch.cuda.reset_peak_memory_stats()

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs is not None:
            elapsed = time.time() - self.t0
            per_step = elapsed / max(state.global_step, 1)
            print(f"[step {state.global_step}/{state.max_steps}] {logs} | {per_step:.1f}s/step "
                  f"| GPU 최대 {torch.cuda.max_memory_allocated() / 2**30:.1f}GiB")


smoke = MAX_STEPS is not None
training_args = TrainingArguments(
    output_dir=str(output_dir),
    per_device_train_batch_size=1,
    gradient_accumulation_steps=32,
    per_device_eval_batch_size=1,
    num_train_epochs=2,
    max_steps=MAX_STEPS or -1,
    learning_rate=2e-4,
    warmup_steps=10,
    bf16=True,
    logging_steps=5,
    eval_strategy="no" if smoke else "steps",
    eval_steps=50,
    save_strategy="no" if smoke else "steps",
    save_steps=50,
    save_total_limit=3,
    load_best_model_at_end=not smoke,
    metric_for_best_model="eval_loss",
    report_to="none",
    prediction_loss_only=True,     # eval에서 logits를 모아 두지 않는다
    gradient_checkpointing=True,
    train_sampling_strategy="group_by_length",
    length_column_name="length",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=collate,
    compute_loss_func=tail_loss,
    callbacks=[PrintLossCallback()],
)

train_result = trainer.train()
print(train_result)

Step,Training Loss,Validation Loss
50,0.499548,0.439082
100,0.423438,0.407207
150,0.381800,0.392143
200,0.407034,0.389750
250,0.358435,0.383645
300,0.271602,0.382223
302,0.271602,0.382147


[step 5/302] {'loss': 1.255897331237793, 'grad_norm': 1.5010573863983154, 'learning_rate': 8e-05, 'epoch': 0.03325711910205779} | 64.2s/step | GPU 최대 9.9GiB
[step 10/302] {'loss': 0.7353758335113525, 'grad_norm': 0.5655831694602966, 'learning_rate': 0.00018, 'epoch': 0.06651423820411557} | 53.1s/step | GPU 최대 9.9GiB
[step 15/302] {'loss': 0.6767783164978027, 'grad_norm': 0.35181304812431335, 'learning_rate': 0.00019726027397260273, 'epoch': 0.09977135730617336} | 46.7s/step | GPU 최대 9.9GiB
[step 20/302] {'loss': 0.5731714725494385, 'grad_norm': 0.30036860704421997, 'learning_rate': 0.00019383561643835617, 'epoch': 0.13302847640823115} | 41.6s/step | GPU 최대 9.9GiB
[step 25/302] {'loss': 0.49249019622802737, 'grad_norm': 0.26453039050102234, 'learning_rate': 0.00019041095890410961, 'epoch': 0.16628559551028893} | 37.8s/step | GPU 최대 9.9GiB
[step 30/302] {'loss': 0.4765209197998047, 'grad_norm': 0.301156610250473, 'learning_rate': 0.00018698630136986303, 'epoch': 0.1995427146123467} | 34.

In [5]:
# ===== 5: 어댑터 저장 =====
if MAX_STEPS is None:
    trainer.save_model(str(output_dir))
    tokenizer.save_pretrained(str(output_dir))
    print(f"어댑터 저장 완료: {output_dir}")
else:
    print("MAX_STEPS 확인 실행 — 저장하지 않음")

어댑터 저장 완료: /home/kopo17/AI_programming/petmint-rag-chatbot/models/lora_adapter/rag-1_4811건


In [6]:
# ===== 6: val 샘플로 답변 확인 (종류별 1건씩) =====
# 추론 메시지는 학습과 같은 prompts.build_messages 형식 — val 샘플의 system·user를 그대로 쓴다
model.eval()


def generate(messages: list[dict], max_new_tokens: int = 300) -> str:
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt", add_special_tokens=False).to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False,
                             pad_token_id=tokenizer.pad_token_id)
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()


shown = set()
for r in val_records:
    kind = r["meta"]["kind"]
    if kind in shown:
        continue
    shown.add(kind)
    q = r["messages"][1]["content"].split(prompts.QUESTION_HEADER + "\n")[-1]
    print(f"\n=== [{kind}] 발췌 {r['meta']['n_items']}개, 정답 위치 {r['meta']['gold_pos']} ===")
    print("질문:", q)
    print("목표:", r["messages"][2]["content"])
    print("생성:", generate(r["messages"][:2]))


=== [distract] 발췌 3개, 정답 위치 1 ===
질문: 이 보험계약의 주된 목적은 무엇인가요?
목표: 이 보험계약의 주된 목적은 피보험자의 상해에 대한 위험을 보장하는 것입니다.
생성: 이 보험계약의 주된 목적은 피보험자의 상해에 대한 위험을 보장하는 것입니다.

=== [gold_only] 발췌 1개, 정답 위치 0 ===
질문: ‘뚜렷한 시야 장해’란 무엇을 의미하며, 어떻게 평가하나요?
목표: ‘뚜렷한 시야 장해’는 한 눈의 시야 범위가 정상시야 범위의 60% 이하로 제한된 경우를 말합니다. 이는 공인된 시야검사방법으로 측정되며, 자동시야검사계(골드만 시야검사)를 이용해 8방향 시야범위 합계를 정상범위와 비교하여 평가합니다.
생성: ‘뚜렷한 시야 장해’는 한 눈의 시야 범위가 정상시야 범위의 60% 이하로 제한된 경우를 말합니다. 이 경우 공인된 시야검사방법으로 측정하며, 자동시야검사계(골드만 시야검사)를 이용해 8방향 시야범위 합계를 정상범위와 비교하여 평가합니다.

=== [no_answer] 발췌 2개, 정답 위치 None ===
질문: 이 특별약관은 어떤 상황에서 적용될 수 없나요?
목표: 제공된 약관 발췌에서는 확인되지 않는 내용입니다. 정확한 안내를 위해 추가 확인이 필요합니다.
생성: 이 특별약관은 모든 피보험자 또는 모든 보험수익자가 장애인이 아닌 경우, 그리고 현재 법정상속인이 장애인이라고 하더라도 수익자 지정이 필요한 경우에 적용될 수 없습니다.
